# 🧠 Notebook 02 — Maximum A Posteriori (MAP)

**Mục tiêu:**
- Hiểu tại sao cần thêm prior vào MLE
- Xem prior "kéo" ước lượng như thế nào
- Trực quan hóa posterior = likelihood × prior

**Điều kiện tiên quyết:** Đã xem notebook `01_mle_intro.ipynb`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.mle import BernoulliMLE, GaussianMLE
from src.map import BernoulliMAPEstimator, GaussianMAPEstimator
from src.utils import make_gaussian, make_bernoulli

plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ Import thành công!')

---
## Phần 1: Vấn đề của MLE khi n nhỏ

Tưởng tượng bạn tung đồng xu **3 lần** và cả 3 lần đều ngửa.

MLE sẽ nói: `p = 1.0` — "đồng xu sẽ luôn luôn ngửa!"

Điều này có hợp lý không? Rõ ràng là không. **MAP giải quyết vấn đề này.**

In [ ]:
data_extreme = [1, 1, 1]  # 3 lần ngửa liên tiếp

p_range = np.linspace(0.001, 0.999, 500)

# Likelihood
k, n = sum(data_extreme), len(data_extreme)
likelihood = p_range**k * (1 - p_range)**(n - k)
likelihood /= likelihood.max()  # normalize để vẽ đẹp

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(p_range, likelihood, 'steelblue', lw=2.5, label='Likelihood L(p | data)')
ax.axvline(1.0, color='red', lw=2.5, linestyle='--', label='MLE: p = 1.0 ⚠️')
ax.fill_between(p_range, likelihood, alpha=0.15, color='steelblue')
ax.set_xlabel('p (xác suất ngửa)')
ax.set_ylabel('Likelihood (đã normalize)')
ax.set_title('Data = [1, 1, 1] — Likelihood tăng mãi đến p=1.0')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

print('⚠️  MLE = 1.0 có nghĩa là "đồng xu sẽ mãi mãi ngửa" — quá cực đoan!')
print('📌 Chúng ta cần thêm kiến thức trước (prior) để điều tiết.')

---
## Phần 2: Prior là gì?

**Prior = niềm tin của bạn về tham số TRƯỚC khi nhìn vào data.**

Với đồng xu thông thường, bạn sẽ tin `p ≈ 0.5`. Ta dùng phân phối **Beta(α, β)** để biểu diễn niềm tin này.

- `Beta(1, 1)` → không có niềm tin gì, mọi p đều như nhau (uniform)
- `Beta(2, 2)` → nhẹ nhàng tin p ≈ 0.5
- `Beta(10, 10)` → rất tin p ≈ 0.5

In [ ]:
p_range = np.linspace(0.001, 0.999, 500)
priors = [
    (1, 1,   'Beta(1,1) — uniform, không có prior'),
    (2, 2,   'Beta(2,2) — nhẹ nhàng tin p≈0.5'),
    (5, 5,   'Beta(5,5) — khá tin p≈0.5'),
    (10, 10, 'Beta(10,10) — rất tin p≈0.5'),
    (5, 2,   'Beta(5,2) — tin p hơi cao'),
]

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['gray', '#3498db', '#2ecc71', '#e74c3c', '#e67e22']

for (a, b, label), color in zip(priors, colors):
    ax.plot(p_range, stats.beta.pdf(p_range, a, b), 
            lw=2, label=label, color=color)

ax.set_xlabel('p')
ax.set_ylabel('Mật độ xác suất')
ax.set_title('Các dạng Beta prior — α, β càng lớn, prior càng "chắc chắn"')
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

---
## Phần 3: MAP = Likelihood × Prior

**Bayes' theorem:**
```
Posterior ∝ Likelihood × Prior
P(p | data) ∝ P(data | p) × P(p)
```

MAP tìm đỉnh (mode) của posterior.

In [ ]:
def plot_map_components(data, alpha, beta, ax, title=''):
    """Vẽ prior, likelihood, và posterior cùng nhau."""
    k = sum(data)
    n = len(data)
    p_range = np.linspace(0.001, 0.999, 500)
    
    # Prior
    prior = stats.beta.pdf(p_range, alpha, beta)
    prior /= prior.max()
    
    # Likelihood
    lik = p_range**k * (1 - p_range)**(n - k)
    lik /= lik.max() if lik.max() > 0 else 1
    
    # Posterior = Beta(α+k, β+n-k)
    post = stats.beta.pdf(p_range, alpha + k, beta + (n - k))
    post /= post.max()
    
    p_mle = k / n
    p_map = (alpha + k - 1) / (alpha + beta + n - 2) if (alpha + beta + n - 2) > 0 else p_mle
    
    ax.plot(p_range, prior, 'gray', lw=1.5, linestyle='--', label=f'Prior Beta({alpha},{beta})', alpha=0.7)
    ax.plot(p_range, lik, 'steelblue', lw=2, label='Likelihood', alpha=0.8)
    ax.plot(p_range, post, '#e74c3c', lw=2.5, label=f'Posterior Beta({alpha+k},{beta+n-k})')
    ax.axvline(p_mle, color='steelblue', lw=1.5, linestyle=':', alpha=0.8)
    ax.axvline(p_map, color='#e74c3c', lw=1.5, linestyle=':')
    ax.text(p_mle, 0.6, f'MLE\n{p_mle:.2f}', color='steelblue', ha='center', fontsize=8)
    ax.text(p_map + 0.06, 0.85, f'MAP\n{p_map:.2f}', color='#e74c3c', ha='center', fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.15)
    ax.set_xlabel('p')

# Cùng data [1,1,1], 3 loại prior khác nhau
data_ex = [1, 1, 1]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

plot_map_components(data_ex, 1, 1, axes[0], 'Prior Beta(1,1) — uniform\n(MAP = MLE)')
plot_map_components(data_ex, 2, 2, axes[1], 'Prior Beta(2,2) — nhẹ\n(MAP kéo về 0.5)')
plot_map_components(data_ex, 5, 5, axes[2], 'Prior Beta(5,5) — mạnh\n(MAP kéo nhiều hơn)')

axes[0].legend(fontsize=8, loc='upper left')
plt.suptitle('Data = [1,1,1] — Prior càng mạnh, MAP càng xa MLE', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Phần 4: Hiệu ứng cỡ mẫu — MAP hội tụ về MLE khi n lớn

In [ ]:
TRUE_P = 0.7
ALPHA, BETA = 2, 2  # prior nhẹ

rng = np.random.default_rng(seed=42)
big_data = rng.binomial(1, TRUE_P, size=500)

n_values = [3, 10, 30, 100, 300, 500]
p_mle_list, p_map_list = [], []

for n in n_values:
    subset = big_data[:n].tolist()
    est_mle = BernoulliMLE().fit(subset)
    est_map = BernoulliMAPEstimator(alpha=ALPHA, beta=BETA).fit(subset)
    p_mle_list.append(est_mle.p)
    p_map_list.append(est_map.p)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(n_values, p_mle_list, 'o-', color='steelblue', lw=2, ms=7, label='MLE')
ax.plot(n_values, p_map_list, 's-', color='#e74c3c', lw=2, ms=7, label=f'MAP (prior Beta({ALPHA},{BETA}))')
ax.axhline(TRUE_P, color='green', lw=2, linestyle='--', label=f'p thật = {TRUE_P}')
ax.axhline(ALPHA / (ALPHA + BETA), color='orange', lw=1.5, linestyle=':', 
           alpha=0.7, label=f'Prior mean = {ALPHA/(ALPHA+BETA)}')

# Annotate differences
for n, mle, map_ in zip(n_values, p_mle_list, p_map_list):
    ax.annotate(f'Δ={abs(mle-map_):.2f}', xy=(n, (mle+map_)/2),
               xytext=(5, 0), textcoords='offset points', fontsize=7, color='gray')

ax.set_xlabel('Cỡ mẫu n')
ax.set_ylabel('Ước lượng p')
ax.set_title('Khi n tăng: MAP → MLE (data thắng prior)')
ax.legend()
ax.set_ylim(0.3, 1.05)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

print(f'📌 Khi n nhỏ: MLE và MAP khác nhau nhiều (prior ảnh hưởng mạnh)')
print(f'📌 Khi n lớn: MAP ≈ MLE (data áp đảo prior)')

---
## Phần 5: MAP cho Gaussian — Gaussian prior

Với likelihood Gaussian và Gaussian prior, MAP cũng có nghiệm dạng đóng đẹp:

$$\mu_{MAP} = \frac{\frac{n}{\sigma^2}\bar{x} + \frac{1}{\tau^2}\mu_0}{\frac{n}{\sigma^2} + \frac{1}{\tau^2}}$$

Đây là **trung bình có trọng số** giữa data mean và prior mean, trọng số là precision.

In [ ]:
TRUE_MU = 3.0
SIGMA = 2.0  # likelihood std (assumed known)

# Prior: μ ~ N(0, τ²) — tin ban đầu μ ≈ 0
MU0, TAU = 0.0, 1.5

sample_sizes = [2, 5, 20, 100]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

mu_range = np.linspace(-3, 7, 400)

for ax, n in zip(axes, sample_sizes):
    data_n = make_gaussian(n=n, mu=TRUE_MU, sigma=SIGMA, seed=n*7)
    
    mle_est = GaussianMLE().fit(data_n)
    map_est = GaussianMAPEstimator(mu0=MU0, tau=TAU, sigma=SIGMA).fit(data_n)
    
    # Prior
    prior = stats.norm.pdf(mu_range, MU0, TAU)
    prior /= prior.max()
    
    # Posterior precision
    post_precision = n / SIGMA**2 + 1 / TAU**2
    post_std = 1 / np.sqrt(post_precision)
    posterior = stats.norm.pdf(mu_range, map_est.mu, post_std)
    posterior /= posterior.max()
    
    ax.plot(mu_range, prior, 'gray', lw=1.5, linestyle='--', 
            label=f'Prior N({MU0},{TAU}²)', alpha=0.7)
    ax.plot(mu_range, posterior, '#e74c3c', lw=2.5, label='Posterior')
    ax.axvline(mle_est.mu, color='steelblue', lw=2, linestyle=':', 
               label=f'MLE: {mle_est.mu:.2f}')
    ax.axvline(map_est.mu, color='#e74c3c', lw=2, linestyle='-.',
               label=f'MAP: {map_est.mu:.2f}')
    ax.axvline(TRUE_MU, color='green', lw=1.5, linestyle='--', alpha=0.6,
               label=f'Thật: {TRUE_MU}')
    
    ax.set_title(f'n = {n}  |  MLE={mle_est.mu:.2f}, MAP={map_est.mu:.2f}', fontsize=10)
    ax.set_xlabel('μ')
    ax.set_xlim(-3, 7)
    if n == 2: ax.legend(fontsize=7.5)

plt.suptitle(f'MAP Gaussian — Prior N({MU0},{TAU}²), μ thật = {TRUE_MU}', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('📌 n=2: MAP bị kéo mạnh về prior (MU0=0)')
print('📌 n=100: MAP ≈ MLE, posterior hẹp lại và tập trung vào μ thật')

---
## Tóm tắt

| | MLE | MAP |
|---|---|---|
| **Công thức** | argmax log P(data\|θ) | argmax [log P(data\|θ) + log P(θ)] |
| **Bernoulli** | k/n | (α+k−1)/(α+β+n−2) |
| **Gaussian μ** | mean(data) | trung bình có trọng số |
| **n nhỏ** | Không ổn định | Ổn định hơn nhờ prior |
| **n lớn** | Tốt | ≈ MLE |

**Bước tiếp theo:** Làm bài tập trong `exercises/beginner/ex03_map_bernoulli.py`, sau đó xem `03_comparison.ipynb`